In [1]:
%pip install pulp
import pulp 


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:


def optimizar_camiones(prediccion_volumen, mae_modelo, factor_estiba=0.85):
    volumen_necesario = (prediccion_volumen + mae_modelo) / factor_estiba
    problema = pulp.LpProblem("Optimizar_Flota_Olist", pulp.LpMinimize)
    
    # 1er Error corregido: "lowBound" lleva la primera "l" en minúscula
    camion_pequeño = pulp.LpVariable("Pequeño_1500kg", lowBound=0, cat='Integer')
    camion_mediano = pulp.LpVariable("Mediano_6000kg", lowBound=0, cat='Integer')
    camion_grande = pulp.LpVariable("Grande_14000kg", lowBound=0, cat='Integer')
    
    # Función objetivo 
    problema += 350*camion_pequeño + 900*camion_mediano + 1800*camion_grande, "Costo_Total"

    # Restricciones de capacidad
    problema += 1500* camion_pequeño + 6000*camion_mediano + 14000*camion_grande >= volumen_necesario, "Capacidad_minima"

    # 2do Error corregido: El solver se escribe "PULP_CBC_CMD" (tenías PUPL)
    problema.solve(pulp.PULP_CBC_CMD(msg=False))

    print(f"Prediccion del ML: {prediccion_volumen} kg/vol")
    print(f"Volumen a cubrir: {volumen_necesario:.2f} kg/vol")
    print("-" * 30)
    print(f"Contratar Pequeños: {int(camion_pequeño.varValue)} camiones")
    print(f"Contratar Medianos: {int(camion_mediano.varValue)} camiones")
    print(f"Contratar Grandes: {int(camion_grande.varValue)} camiones")
    print("-" * 30)
    
    # 3er Error corregido: La propiedad en inglés es "objective" (con "c")
    print(f"Costo Total Optimizado: R$ {pulp.value(problema.objective)}")
    


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
import xgboost as xgb

# 1. Cargar el dataset directamente en esta celda
df = pd.read_csv('../data/dataset_rutas_limpios.csv')

# 2. Convertir fecha a índice si existe la columna
if 'fecha' in df.columns:
    df.set_index('fecha', inplace=True)

# 3. Separar X e y, y convertir texto a números
X = df.drop(columns=['volumen_total_cm3'])
y = df['volumen_total_cm3']
X = pd.get_dummies(X, drop_first=True)

# 4. Crear X_train y y_train AQUÍ MISMO
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 5. Crear, entrenar y predecir
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

xgb_model.fit(X_train, y_train)
predicciones_xgb = xgb_model.predict(X_test)



In [4]:
from sklearn.metrics import mean_absolute_error

# 1. Calculamos el MAE real de tu modelo XGBoost
mae_xgb = mean_absolute_error(y_test, predicciones_xgb)

# 2. Corremos la optimización para el primer registro
optimizar_camiones(prediccion_volumen=predicciones_xgb[0], mae_modelo=mae_xgb)

Prediccion del ML: 532718.4375 kg/vol
Volumen a cubrir: 684074.81 kg/vol
------------------------------
Contratar Pequeños: 0 camiones
Contratar Medianos: 0 camiones
Contratar Grandes: 49 camiones
------------------------------
Costo Total Optimizado: R$ 88200.0
